# 08 — Heterogeneous ensemble, confirmation, and deployment validation

This notebook is the employer-facing summary of the post-Stage-43 research frontier. It intentionally contains only aggregate, public-safe evidence: no MRI, study identifiers, row-level predictions, cloud credentials, model weights, or private return bundles.

The internal model-selection metric is scanner-grouped macro ROC-AUC. Public leaderboard scores are reported separately and are **not treated as directly comparable** to grouped OOF values.

In [1]:
import pandas as pd
import plotly.express as px

progression = pd.DataFrame({
    "Milestone": ["Stage 43 DINO", "Stage 47 DINO + MaxViT", "Stage 50 confirmed overlay"],
    "Grouped OOF macro-AUC": [0.7865573869, 0.7912075115, 0.7936872288],
})
fig = px.line(progression, x="Milestone", y="Grouped OOF macro-AUC", markers=True, title="Owned grouped-OOF progression")
fig.update_layout(height=430)
fig.show();

## Stage 50 — independent confirmation

The two-target MCL + Lateral Meniscus overlay was frozen from screening folds 0 and 2 and then evaluated on folds 1, 3, and 4. It improved all three confirmation folds and changed no other targets by design.

Confirmation baseline: **0.795488**  
Confirmation candidate: **0.798017**  
Macro-AUC gain: **+0.002529**

The five-fold 0.793687 result above is descriptive because it includes the screening folds; the three-fold confirmation is the independent evidence for the overlay.

In [2]:
confirmation = pd.DataFrame({
    "Fold": ["1", "3", "4"],
    "Macro-AUC gain": [0.0020505557, 0.0049880294, 0.0014276970],
})
fig = px.bar(confirmation, x="Fold", y="Macro-AUC gain", text="Macro-AUC gain", title="Stage 50 confirmation: gain on every held-out fold")
fig.update_traces(texttemplate="%{text:.4f}", textposition="outside")
fig.update_layout(height=430)
fig.show();

## Deployment validation

Subsequent AWS validation established sampled checkpoint parity across all 15 owned DINO / MaxViT / expert checkpoints, exact sampled raw-image reconstruction on three complete canary studies, and a two-study raw-input smoke through the full owned ensemble.

These checks validate the owned inference chain on bounded samples. They do **not** establish hidden-test runtime, complete test-set series selection, or integration with the stronger scored public reference. Those remain explicit deployment gates.

In [3]:
stage54 = pd.DataFrame({
    "Variant": ["Frozen head", "Legacy-depth head", "Corrected-depth head", "Stage-50 incumbent", "Legacy-depth overlay", "Corrected-depth overlay"],
    "Fold-0 macro-AUC": [0.7477983416, 0.7488270839, 0.7485985661, 0.7880454991, 0.7879493146, 0.7880827269],
})
fig = px.bar(stage54, x="Variant", y="Fold-0 macro-AUC", text="Fold-0 macro-AUC", title="Stage 54 depth-position experiment: valid early rejection")
fig.update_traces(texttemplate="%{text:.4f}", textposition="outside")
fig.update_layout(height=470, xaxis_tickangle=-22)
fig.show();

## Stage 54 decision

The corrected relative-depth head did **not** beat the matched legacy-depth head on fold 0, so a condition required to hold on every screening fold failed. The direction was closed early instead of spending another fold of GPU time.

This is a useful negative result: the raw-input and feature-bank infrastructure remains reusable, while the representation hypothesis is rejected under the prespecified gate.

## Current frontier

The next portfolio-relevant work is to recover and validate the complete scored-reference identity, freeze a scientifically justified integration with the confirmed Stage-50 experts, and complete test-like output/schema/runtime validation before another submission. Further model research should add a materially new capability rather than repeat the rejected depth-head adjustment.